In [1]:
import sys
from pathlib import Path

# horizon/ pipeline modules use flat imports (`from fds.fd import ...`) and need
# horizon/ on sys.path; eval + package imports (`horizon.fds.fd`) need the repo
# root. Put both on the path, repo root first. See app_helpers._bootstrap_path.
REPO = Path("..").resolve()
HZN = REPO / "horizon"
for p in (str(HZN), str(REPO)):
    if p in sys.path:
        sys.path.remove(p)
sys.path.insert(0, str(HZN))
sys.path.insert(0, str(REPO))

from horizon.utils.loaders import get_fds, load_table
from eval.dataset_eval import characterize_dataset
from eval.fd_eval import characterize_fds
from eval.report import characterize

In [2]:
fds_path = Path("..") / "datasets" / "insurance_claims_58k" / "fds.csv"
csv_path = Path("..") / "datasets" / "insurance_claims_58k" / "clean.csv"

In [3]:
fds = get_fds(fds_path, csv_path)
print(f"Loaded {len(fds)} FDs")
fds

Loaded 11 FDs


[FD(model -> engine_type), FD(model -> airbags), FD(model -> width), FD(model -> length), FD(model -> is_power_steering), FD(model -> max_power), FD(engine_type -> displacement), FD(engine_type -> fuel_type), FD(region_code -> region_density), FD(max_torque -> max_power), FD(max_power -> max_torque)]

In [4]:
df = load_table(csv_path)
df.head()

policy_id,subscription_length,vehicle_age,customer_age,region_code,region_density,segment,model,fuel_type,max_torque,max_power,engine_type,airbags,is_esc,is_adjustable_steering,is_tpms,is_parking_sensors,is_parking_camera,rear_brakes_type,displacement,cylinder,transmission_type,steering_type,turning_radius,length,width,gross_weight,is_front_fog_lights,is_rear_window_wiper,is_rear_window_washer,is_rear_window_defogger,is_brake_assist,is_power_door_locks,is_central_locking,is_power_steering,is_driver_seat_height_adjustable,is_day_night_rear_view_mirror,is_ecw,is_speed_alert,ncap_rating,claim_status
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""POL045360""","""9.3""","""1.2""","""41""","""C8""","""8794""","""C2""","""M4""","""Diesel""","""250Nm@2750rpm""","""113.45bhp@4000rpm""","""1.5 L U2 CRDi""","""6""","""Yes""","""Yes""","""Yes""","""Yes""","""Yes""","""Disc""","""1493""","""4""","""Automatic""","""Power""","""5.2""","""4300""","""1790""","""1720""","""Yes""","""Yes""","""Yes""","""Yes""","""Yes""","""Yes""","""Yes""","""Yes""","""Yes""","""No""","""Yes""","""Yes""","""3""","""0"""
"""POL016745""","""8.2""","""1.8""","""35""","""C2""","""27003""","""C1""","""M9""","""Diesel""","""200Nm@1750rpm""","""97.89bhp@3600rpm""","""i-DTEC""","""2""","""No""","""Yes""","""No""","""Yes""","""Yes""","""Drum""","""1498""","""4""","""Manual""","""Electric""","""4.9""","""3995""","""1695""","""1051""","""Yes""","""No""","""No""","""Yes""","""No""","""Yes""","""Yes""","""Yes""","""Yes""","""Yes""","""Yes""","""Yes""","""4""","""0"""
"""POL007194""","""9.5""","""0.2""","""44""","""C8""","""8794""","""C2""","""M4""","""Diesel""","""250Nm@2750rpm""","""113.45bhp@4000rpm""","""1.5 L U2 CRDi""","""6""","""Yes""","""Yes""","""Yes""","""Yes""","""Yes""","""Disc""","""1493""","""4""","""Automatic""","""Power""","""5.2""","""4300""","""1790""","""1720""","""Yes""","""Yes""","""Yes""","""Yes""","""Yes""","""Yes""","""Yes""","""Yes""","""Yes""","""No""","""Yes""","""Yes""","""3""","""0"""
"""POL018146""","""5.2""","""0.4""","""44""","""C10""","""73430""","""A""","""M1""","""CNG""","""60Nm@3500rpm""","""40.36bhp@6000rpm""","""F8D Petrol Engine""","""2""","""No""","""No""","""No""","""Yes""","""No""","""Drum""","""796""","""3""","""Manual""","""Power""","""4.6""","""3445""","""1515""","""1185""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""","""No""","""No""","""No""","""Yes""","""0""","""0"""
"""POL049011""","""10.1""","""1""","""56""","""C13""","""5410""","""B2""","""M5""","""Diesel""","""200Nm@3000rpm""","""88.77bhp@4000rpm""","""1.5 Turbocharged Revotorq""","""2""","""No""","""Yes""","""No""","""Yes""","""No""","""Drum""","""1497""","""4""","""Manual""","""Electric""","""5""","""3990""","""1755""","""1490""","""No""","""No""","""No""","""No""","""No""","""Yes""","""Yes""","""Yes""","""No""","""No""","""Yes""","""Yes""","""5""","""0"""


In [5]:
# drop FDs whose LHS redundancy is below MIN_REDUNDANCY — too few repeats to
# support a pattern (avg LHS group size ≈ n^MIN_REDUNDANCY).
from eval.fd_eval import fd_lhs_redundancy

MIN_REDUNDANCY = 0.1  # ~3 rows per LHS group at n≈170k

red = fd_lhs_redundancy(df, fds)
kept = [fd for fd in fds if red[fd.lhs] >= MIN_REDUNDANCY]
dropped = [fd for fd in fds if red[fd.lhs] < MIN_REDUNDANCY]
print(f"kept {len(kept)} / {len(fds)} FDs; dropping {len(dropped)} (red < {MIN_REDUNDANCY}):")
for fd in dropped:
    print(f"  {red[fd.lhs]:.3f}  {fd}")

# keep the filtered set in memory; don't overwrite the source fds.csv
fds = kept

kept 11 / 11 FDs; dropping 0 (red < 0.1):


In [6]:
# FD-side metrics only make sense when the FDs were defined for *this* table.
# Proxy for "they belong together": both files live in the same dataset folder.
from pprint import pprint

if fds_path.parent == csv_path.parent:
    pprint(characterize(df, fds), sort_dicts=False)
else:
    print(
        f"skipped FDs: {fds_path.parent.name}/fds.csv and {csv_path.parent.name}/clean.csv are from different datasets"
    )
    pprint(characterize_dataset(df), sort_dicts=False)
    pprint(characterize_fds(fds), sort_dicts=False)

{'n_rows': 58592,
 'n_cols': 41,
 'avg_redundancy': 0.8414892206481975,
 'redundancy_per_column': {'policy_id': 1.0,
                           'subscription_length': 418.51428571428573,
                           'vehicle_age': 1195.7551020408164,
                           'customer_age': 1429.0731707317073,
                           'region_code': 2663.2727272727275,
                           'region_density': 2663.2727272727275,
                           'segment': 9765.333333333334,
                           'model': 5326.545454545455,
                           'fuel_type': 19530.666666666668,
                           'max_torque': 6510.222222222223,
                           'max_power': 6510.222222222223,
                           'engine_type': 5326.545454545455,
                           'airbags': 19530.666666666668,
                           'is_esc': 29296.0,
                           'is_adjustable_steering': 29296.0,
                           'is_tpms': 29296

In [7]:
# for a given FD, show each distinct LHS tuple once with its occurrence count,
# sorted by the LHS columns so duplicates / outliers are easy to scan.

from eval.fd_eval import fd_lhs_redundancy

red = fd_lhs_redundancy(df, fds)

def lhs_view(fd, *, n=30):
    # fd.lhs is a bare str for a singleton LHS, a tuple for composite
    cols = list(fd.lhs) if isinstance(fd.lhs, tuple) else [fd.lhs]
    return df.group_by(cols).len(name="count").sort(cols).head(n)

i = 7
fd = fds[i]
print(f"[{i}] {fd}   lhs_redundancy = {red[fd.lhs]:.3f}")
lhs_view(fd, n=30)

[7] FD(engine_type -> fuel_type)   lhs_redundancy = 0.782


engine_type,count
str,u32
"""1.0 SCe""",2373
"""1.2 L K Series Engine""",2940
"""1.2 L K12N Dualjet""",1080
"""1.5 L U2 CRDi""",14018
"""1.5 Turbocharged Revotorq""",1598
…,…
"""F8D Petrol Engine""",14948
"""G12B""",1209
"""K Series Dual jet""",13776
